# TripMe AI Mode - Fine-tune a Sinhala-capable Llama-3 on Sri Lanka places data (Kaggle version)

Runs on a free Kaggle GPU (P100 or T4x2, 16GB+). Fine-tunes `ihalage/llama3-sinhala` - a Llama-3-8B-Instruct model already adapted for Sinhala - with QLoRA on the TripMe places instruction dataset, in **both English and Sinhala**, so the resulting model can hold the same conversation in either language.

**Why this base model:** the source places dataset itself has no Sinhala text (only English `description` fields), so a generic base model (Phi-3, vanilla Llama-3, Gemma) would need to learn Sinhala from scratch during this fine-tune - unrealistic with ~4k examples per language. `ihalage/llama3-sinhala` already understands and generates Sinhala; this fine-tune's job is narrower: teach it the TripMe voice and ground it in the Sri Lanka places data, in both languages.

**Before running:**
1. Create a Kaggle Dataset named `tripme-data` containing `train.jsonl`, `val.jsonl`, `train_si.jsonl`, `val_si.jsonl` (from `data/training/` in the project).
2. Attach that dataset to this notebook (right sidebar > Add Input > search `tripme-data`).
3. In the right sidebar under **Settings**, set **Accelerator** to a GPU (P100, or T4 x2) and **Persistence** to "Files only" so checkpoints survive if the session restarts.

Unlike Colab, there's no Drive mount step - Kaggle mounts your attached dataset automatically at `/kaggle/input/tripme-data/`, and this notebook's own output directory `/kaggle/working/` persists across a session and downloads as a zip via "Save Version" when you're done.</cell-0>


In [ ]:
import os

# Kaggle nests attached datasets under a path that includes your username
# (e.g. /kaggle/input/datasets/<username>/tripme-data), not always the flat
# /kaggle/input/tripme-data/ shape - search for wherever train.jsonl actually
# landed rather than hardcoding one specific path shape.
KAGGLE_INPUT_DIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train.jsonl" in files:
        KAGGLE_INPUT_DIR = root
        break

assert KAGGLE_INPUT_DIR is not None, (
    "Could not find train.jsonl anywhere under /kaggle/input - attach the "
    "'tripme-data' dataset via the right sidebar (Add Input) before running "
    "the rest of this notebook."
)
print("Using data directory:", KAGGLE_INPUT_DIR)
print("Found:", os.listdir(KAGGLE_INPUT_DIR))

In [ ]:
.
!pip install -q -U transformers accelerate peft bitsandbytes datasets
!pip install -q "trl==1.6.0"

In [ ]:
DATA_DIR = KAGGLE_INPUT_DIR
CHECKPOINT_DIR = "/kaggle/working/tripme-checkpoints"
OUTPUT_DIR = "/kaggle/working/tripme-adapter"
BASE_MODEL = "ihalage/llama3-sinhala"  # Llama-3-8B-Instruct fine-tuned for Sinhala (Apache 2.0)
MAX_SEQ_LENGTH = 320  # dataset responses run ~25-115 words (max ~154); 320 tokens covers system+user+assistant with safe headroom, faster than 512

In [ ]:
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

In [ ]:
# Llama-3 attention/MLP module names (different from Phi-3's qkv_proj/gate_up_proj naming)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",       # MLP
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Load and combine English + Sinhala datasets

Both language datasets are trained together in one pass, so the model learns to answer in whichever language it's addressed in, using the same TripMe voice and the same underlying place facts.

In [ ]:
dataset_en = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/train.jsonl",
        "validation": f"{DATA_DIR}/val.jsonl",
    },
)
dataset_si = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/train_si.jsonl",
        "validation": f"{DATA_DIR}/val_si.jsonl",
    },
)

train_combined = concatenate_datasets([dataset_en["train"], dataset_si["train"]]).shuffle(seed=42)
val_combined = concatenate_datasets([dataset_en["validation"], dataset_si["validation"]]).shuffle(seed=42)

# Training in stages: TRAIN_SUBSET_SIZE takes the first N examples of a
# fixed shuffle (seed=42), so raising it later (1500 -> 3000 -> ...) always
# keeps the same earlier examples in the same position and appends new ones
# after them. Combined with resume_from_checkpoint in the train cell, each
# stage only trains on the newly-added examples, not the earlier ones again.
TRAIN_SUBSET_SIZE = 1500
train_combined = train_combined.select(range(min(TRAIN_SUBSET_SIZE, len(train_combined))))

print(f"train: {len(train_combined)} examples (subset of {len(dataset_en['train']) + len(dataset_si['train'])} total)")
print(f"val:   {len(val_combined)} examples ({len(dataset_en['validation'])} en + {len(dataset_si['validation'])} si)")
print(train_combined[0])

In [ ]:
def format_example(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_combined = train_combined.map(format_example, remove_columns=train_combined.column_names)
val_combined = val_combined.map(format_example, remove_columns=val_combined.column_names)
print(train_combined[0]["text"][:500])

In [ ]:
sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=150,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    bf16=True,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_combined,
    eval_dataset=val_combined,
    processing_class=tokenizer,
)

## Train

This is a larger base model (8B) than a Phi-3-mini run would be, so expect a full 3-epoch pass over ~8.4k combined examples to take a few hours. Kaggle sessions have a 9-12 hour limit and can occasionally be interrupted. If training stops partway, re-run the setup cells above (input check, config, model load, LoRA, dataset load, format, SFTConfig), then re-run this cell - it auto-resumes from the latest checkpoint under `/kaggle/working/tripme-checkpoints`, which persists for the life of the session (persistence must be set to "Files only" in the notebook's Settings sidebar - see the note at the top).

In [ ]:
resume = any(d.startswith("checkpoint-") for d in os.listdir(CHECKPOINT_DIR)) \
    if os.path.isdir(CHECKPOINT_DIR) else False

trainer.train(resume_from_checkpoint=resume)

In [ ]:
# Save the LoRA adapter to /kaggle/working/ - click "Save Version" (top right)
# after this finishes to persist it as a downloadable Kaggle Output.
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

## Manual evaluation

Sanity-check the fine-tuned model on realistic queries in both languages before trusting it. Look for: (1) groundedness - does it mention real places/facts from the dataset, not invented ones; (2) tone - does it sound like a warm local guide, not a generic chatbot; (3) does the Sinhala read as natural, grammatically correct formal Sinhala (not stitched/awkward); (4) it should decline or hedge gracefully on questions outside the dataset's scope.

In [ ]:
from transformers import pipeline

gen = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer, max_new_tokens=250)

SYSTEM_PROMPT_EN = (
    "You are TripMe, a warm and knowledgeable Sri Lankan travel voice assistant. "
    "Reply naturally in English, using only the facts provided about each place."
)
SYSTEM_PROMPT_SI = (
'ඔබ TripMe නම් වූ, ශ්\u200dරී ලංකාවේ සංචාරක ස්ථාන පිළිබඳ නිර්දේශ ලබා දෙන කථන සහායකයෙකි. ලබා දී ඇති කරුණු පමණක් භාවිතා කරමින්, පිරිසිදු හා විධිමත් සිංහල භාෂාවෙන් ස්වාභාවිකව හා උණුසුම්ව පිළිතුරු දෙන්න.'
)

eval_prompts = [
    (SYSTEM_PROMPT_EN, "What's worth visiting near Kandy?"),
    (SYSTEM_PROMPT_EN, "Is Dunhinda safe to visit right now?"),
    (SYSTEM_PROMPT_EN, "Can you plan a day trip around Nuwara Eliya for me?"),
    (SYSTEM_PROMPT_EN, "Play the audio guide for Dunhinda."),
    (SYSTEM_PROMPT_EN,
     "My trip: 3 days in Kandy, day 1 just finished (2 days left). Total budget: 35000 LKR. "
     "Spent so far: 8000 LKR (transport 1300, food 3900, tickets 1400, stay 1400).\n\n"
     "Am I on track with my spending?"),
    (SYSTEM_PROMPT_SI, "Kandy ආසන්නයේ නැරඹීමට සුදුසු ස්ථානයක් තිබේද?"),
    (SYSTEM_PROMPT_SI, "Dunhinda දැන් යාමට ආරක්ෂිතද?"),
    (SYSTEM_PROMPT_SI, "Nuwara Eliya අවට එක්දින සංචාරයක් සැලසුම් කර දෙන්නද?"),
    (SYSTEM_PROMPT_SI, "Dunhinda ගැන කථන මාර්ගෝපදේශය අරඹන්න."),
]

for system_prompt, prompt in eval_prompts:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = gen(formatted, do_sample=True, temperature=0.7, top_p=0.9)[0]["generated_text"]
    response = output[len(formatted):].strip()
    print(f"Q: {prompt}\nA: {response}\n{'-'*80}")